# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahid-nawazai/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

My unit of analysis is one pseudonymized content page (content_hash_id) for a specific client (client_hash_id) aggregated over a specific monthly window.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I will use fact_content_daily_performance (for daily search metrics and proxy labels) joined with dim_content (for static page metadata).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I am using a mid-panel month—specifically March 2026 (2026-03)—to build my feature frame. I am deliberately avoiding the final _sample month (June 2026) so it remains a sealed test set for the future window.

In [8]:
import duckdb
import pandas as pd
from google.colab import userdata

# 1. Connect to DuckDB and Authenticate using the new Secrets Manager
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# ---> THIS IS THE NEW LINE <---
con.execute(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# Define the warehouse table path
TABLE_URL = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'

print("--- 1. Row Count & Date Span (March 2026) ---")
q1 = f"""
SELECT
    MIN(report_date) as start_date,
    MAX(report_date) as end_date,
    COUNT(*) as total_rows
FROM '{TABLE_URL}'
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
"""
display(con.execute(q1).df())

print("\n--- 2. The Grain Check ---")
q2 = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) as row_count
FROM '{TABLE_URL}'
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-03'
GROUP BY 1, 2, 3
ORDER BY row_count DESC
LIMIT 5
"""
display(con.execute(q2).df())
print("If row_count is 1, the grain is exactly what we contracted: client + content + date.")

print("\n--- 3. Availability Check (ga4_data_available IS TRUE) ---")
q3 = f"""
SELECT
    COUNT(*) as rows_with_ga4_data
FROM '{TABLE_URL}'
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
AND ga4_data_available IS TRUE
"""
display(con.execute(q3).df())

--- 1. Row Count & Date Span (March 2026) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows
0,2026-03-01,2026-03-31,9841378



--- 2. The Grain Check ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,2026-03-01,1
1,client_62f4a7e64f5e0096,content_39d7361b4945d504,2026-03-01,1
2,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,2026-03-01,1
3,client_62f4a7e64f5e0096,content_4dc944b7d0b65ecc,2026-03-01,1
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,2026-03-01,1


If row_count is 1, the grain is exactly what we contracted: client + content + date.

--- 3. Availability Check (ga4_data_available IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_ga4_data
0,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

I am predicting a proxy label: whether the page shows a declining trend (trend_direction == 'down').

In [13]:
# Peek at all the column names in the warehouse table
columns_df = con.execute(f"DESCRIBE SELECT * FROM '{TABLE_URL}'").df()
print(columns_df['column_name'].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [14]:
print("--- 4. Feature Frame & The Trap ---")
q4 = f"""
SELECT
    client_hash_id,
    content_hash_id,

    -- FEATURE 1: knowable at the decision moment because daily impressions are already recorded.
    gsc_impressions,

    -- FEATURE 2: knowable at the decision moment because daily organic sessions are a historical fact.
    sessions_organic,

    -- FEATURE 3: knowable at the decision moment because the average position is tracked daily.
    gsc_avg_position,

    -- FEATURE 4: knowable at the decision moment because AI-driven sessions are logged after they occur.
    sessions_ai,

    -- FEATURE 5: knowable at the decision moment because total engagement seconds are tracked by GA4 daily.
    ga4_total_engagement_sec,

    -- THE TRAP (Target Leakage): gsc_clicks.
    -- If our proxy label for a "declining page" is based on a drop in clicks, including the exact click count gives the model a direct shortcut to the answer!
    gsc_clicks as leaky_clicks_trap

FROM '{TABLE_URL}'
WHERE report_date = '2026-03-31'
LIMIT 10
"""
features_df = con.execute(q4).df()
display(features_df.head())

print("\nDropping the leaky trap feature to keep the honest numbers...")
clean_features_df = features_df.drop(columns=['leaky_clicks_trap'])
display(clean_features_df.head())

--- 4. Feature Frame & The Trap ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,sessions_organic,gsc_avg_position,sessions_ai,ga4_total_engagement_sec,leaky_clicks_trap
0,client_62f4a7e64f5e0096,content_7cdbe7eb2e6669ca,4,<NA>,2.500000,<NA>,<NA>,0
1,client_62f4a7e64f5e0096,content_bb843e565f31bb7b,237,<NA>,2.227848,<NA>,<NA>,1
2,client_62f4a7e64f5e0096,content_12d1c050115b68a7,129,<NA>,2.333333,<NA>,<NA>,0
3,client_62f4a7e64f5e0096,content_e81b071d5fabc22d,8,<NA>,6.500000,<NA>,<NA>,0
4,client_62f4a7e64f5e0096,content_907167e650250839,104,<NA>,9.519231,<NA>,<NA>,0



Dropping the leaky trap feature to keep the honest numbers...


,client_hash_id,content_hash_id,gsc_impressions,sessions_organic,gsc_avg_position,sessions_ai,ga4_total_engagement_sec
0,client_62f4a7e64f5e0096,content_7cdbe7eb2e6669ca,4,<NA>,2.500000,<NA>,<NA>
1,client_62f4a7e64f5e0096,content_bb843e565f31bb7b,237,<NA>,2.227848,<NA>,<NA>
2,client_62f4a7e64f5e0096,content_12d1c050115b68a7,129,<NA>,2.333333,<NA>,<NA>
3,client_62f4a7e64f5e0096,content_e81b071d5fabc22d,8,<NA>,6.500000,<NA>,<NA>
4,client_62f4a7e64f5e0096,content_907167e650250839,104,<NA>,9.519231,<NA>,<NA>


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.